## DataFrame Functional Basics

- DataFrame operations are automatically distributed across partitions (workers) for parallel processing (speed).  This is why when dataframes are written to disk as csv, parquet, etc, they are written in 'fileparts', unless coalesce into a single file.  Each partition 'writes its part' of the final product.

- Methods return a new dataframe as dataframes are **immutable**.

- Each operation builds upon a logical plan until an 'action' (count, write, etc), triggers an execution

- Dataframe Transformations have equivalent SQL operations under the hood.  For example, the dataframe 'filter' method is equivalent to a WHERE clause in SQL

## DataFrame Transformation Methods


```
- select()
- filter(), where()
- groupBy()
- orderBy(), sort()
- join()
```

## DataFrame Missing Values

![](/Volumes/workspace/pyspark_learning/raw_files/images/missing_values.png)

## How to Reference Data (which approach do I use?)

![](/Volumes/workspace/pyspark_learning/raw_files/images/reference_data.png)

**When you create a schema with python StructTypes they are dot-notation (attribute) accessible because they are valid python types**

### Common uses of `col`

![](/Volumes/workspace/pyspark_learning/raw_files/images/colUse.png)

using 'col' is the most flexible, but where I can I use attribute due to ease of typing it all out

column object methods are methods on the column object.  These are different than functions which take columns as arguments.
Example:


method example, methods are inherent on object
```
col("name").contains("manager")
```

function example, columns are arguments
```
coalesce(col("name"), col("nickname"))
```

## Built-in Functions ##

- Functions operate on entire columns within DataFrames
- Functions may operate on any data type
- There are built in DataFrame and SQL, equivalent functions

## Python Function vs UDF ##

A Python function is a standard block of code that runs locally on a driver or within a single Python process, while a User-Defined Function (UDF) is a specialized wrapper that allows that Python code to execute across a distributed Databricks cluster.

![](/Volumes/workspace/pyspark_learning/raw_files/images/python_vs_udf.png)



#### Pandas UDF vs UDF

![](/Volumes/workspace/pyspark_learning/raw_files/images/pandas_vs_udf.png)

## GroupBy in DataFrames

- groupBy returns a grouped object which then allows for (and requires) some sort of aggregation method to be applied.
- groupBy operations partion data across Worker nodes and are computationally expensive (shuffle)
- Aggregations execute in parallel across these partitions
- groupBy is lazy evaluated until an action triggers an execution


group by does not require `col` expression but it CAN be used, see below.  The actions are 'count', 'avg', and 'sum'

```
df.groupBy("department").count()
```

```
df.groupBy("department", "location").avg("salary")
```

```
df.groupBy(col("department"), year("hire_date)).sum("revenue")
```

Basic Aggregation methods:

```
- count()    - counts all non-null rows in each group
- sum(col)
- avg(col)
- min(col)/max(col)
```

Multiple aggregation methods can be applied to a grouped object via the use of the ```agg()``` function

```
df.groupBy("department").agg(sum("salary"), avg("age")).display()
```

Alternate dictionary syntax
```
df.groupBy("department").agg({
  "salary": "sum",
  "age": "avg"
}).display()
```


![](/Volumes/workspace/pyspark_learning/raw_files/images/group_by.png)

## Basic Aggregation Methods


![](/Volumes/workspace/pyspark_learning/raw_files/images/aggregation.png)

### Combining Aggregations

![](/Volumes/workspace/pyspark_learning/raw_files/images/multiple_agg.png)

## Relational Operations

- **UNION** - Includes all unique elements from both sets (no dupes)
- **UNIONBYNAME** - Combines dataframes by matching column names
- **INTERSECTION** - Includes only the elements present in each set
- **SUBTRACTION** - Includes only the difference from left to right where returns only what is unqiue to left

![](/Volumes/workspace/pyspark_learning/raw_files/images/setoperations.png)

### Intersect Example ###

In [0]:
# Find the overlap - suppliers that exist in both tables
common_suppliers = franchise_suppliers.intersect(all_suppliers)
display(common_suppliers)

### Subtract Example ###


In [0]:
# Identify supplier IDs in each DataFrame
franchise_suppliers = franchises_df.select("supplierID").distinct()
all_suppliers = suppliers_df.select("supplierID").distinct()

# Find supplierIDs that are in franchises_df but not in suppliers_df
franchises_without_valid_suppliers = franchise_suppliers.subtract(all_suppliers)
display(franchises_without_valid_suppliers)

### Joins

- Left:   keeps all rows from the left dataframe along with matches from the right dataframe
- Right:  keeps all rows from the right dataframe along with matches from the left dataframe
- Outer:  keeps all rows from both dataframes, filling nulls where no match exists
- Inner:  keeps only rows which exist in both dataframes
- ALWAYS (where possible), join small to large. syntax:  ```small_df.join(large_df, key)```

## full outer join

In [0]:
# A full outer join returns all records from both DataFrames, matching rows where possible.
# If there is no match, the missing side will contain nulls.

df1 = spark.createDataFrame([
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie")
], ["id", "name"])

df2 = spark.createDataFrame([
    (2, "Sales"),
    (3, "Marketing"),
    (4, "Finance")
], ["item_id", "department"])

# Perform a full outer join on 'id'
# result = df1.join(df2, on="id", how="outer")
result = df1.join(df2, df2.item_id == df1.id, how="outer")
display(result)

In [0]:
# Please note how all fields from both dataframes are present in the result of am inner join, a better practice is to project the columns you need from each entity instead of returning all from all...

# We will also alias some of the columns to disambiguate column names

enriched_transactions = franchises_df \
    .select(
        "franchiseID", 
        col("name").alias("store_name"), 
        col("city").alias("store_city"), 
        col("country").alias("store_country")
        ) \
    .join(
        transactions_df,
        on="franchiseID",
        how="inner"
    )
    
display(enriched_transactions)

## Left_anti Join vs Subtract

In [0]:
# A left_anti join returns only the rows from the left DataFrame that do not have a match in the right DataFrame.  Returns rows unique to the lef

left_anti_result = df1.join(df2, on="id", how="left_anti")
display(left_anti_result)

In [0]:
# left_anti join returns rows from df1 where 'id' does not exist in df2.
left_anti_result = df1.join(df2, on="id", how="left_anti")
display(left_anti_result)

# subtract returns rows from df1 that are not present in df2, considering all columns.
subtract_result = df1.subtract(df2)
display(subtract_result)

## Complex Data - Nested Data

Nested JSON is an example of complex data

![](/Volumes/workspace/pyspark_learning/raw_files/images/complex_data.png)

Below is an example of how to parse/apply schema to nested json.

Notice the MapType() takes a type for both key and value,  essentially a python dict

![](/Volumes/workspace/pyspark_learning/raw_files/images/json.png)


```col("struct_column.field_name')``` or ``` getField()``` allows for direct access to nested data

In [0]:
"""
Example of creating a schema with an array containing strings
"""
from pyspark.sql.types import *

interests_schema = ArrayType(StringType())

In [0]:
"""
Working with StructTypes
Struct fields can accessed using dot notation or the getField() column function
"""

# example of dot notation and getField
# where data has a struct column named 'user'
# user: struct<name:string, age:int, scores:array<double>>

from spark.sql.functions import col
df.select(
    col("user.name"),
    col("user").getField("age"),
    col("user.scores")[0].alias("first_score")
)

# JSON #

Use the *schema_of_json* Function to generate a Schema based upon a sample row of data
In many cases, especially with multiple nested complex structures, it is easiest to generate a schema based upon a sample of JSON data, we can do this using the schema_of_json Function.

In [0]:
"""
Example
"""
from spark.sql.functions import schema_of_json, lit
# Get the schema for the recent_purchases JSON
recent_purchases_schema = schema_of_json(lit(recent_purchases_json))

## Explode
- Explode flattens nested data
- Takes a single row with an array or map and creates a new row for each item within that collection
- Drops Nulls/Empties: Rows where the array or map column is NULL or empty are entirely excluded from the result.
![](/Volumes/workspace/pyspark_learning/raw_files/images/explode.png)

**explode**

The ```explode()``` function unnests data contained within an array

- explode on large data sets is dangerous as it may overwhelm memory when one row becomes 'n' rows

![](/Volumes/workspace/pyspark_learning/raw_files/images/arrayOperations.png)

In [0]:

'''
below is an explode example were the array is broken out into individual records
'''
from pyspark.sql.functions import explode

data = [
    (1, ["a", "b", "c"]),
    (2, ["x", "y"])
]

# ddl schema type
schema = "id integer, items array<string>"

df1 = spark.createDataFrame(data, schema=schema)
df1.display()


In [0]:
"""
Explode the data from df1
NOTE: explode is not a dataframe method but should be used in your `select` statement
"""
df1.select('id', explode("items").alias('item')).display()

In [0]:
"""
Here is a more complex example
where a row has an array containing 'a', explode that array into unique rows
NOTE:  array_contains can be used as an expression in a string OR it can be explictly imported and used a as a function.  Below it is used in a string expression
"""
df1.filter("array_contains(items, 'a')").select("id", explode('items')).display()


# Collect #

In Apache Spark on Databricks, collect() is an action that retrieves all records from a distributed DataFrame or RDD and brings them to the driver node as a local Python list of Row objects.

While convenient for small datasets, it is often discouraged for large-scale data because it can easily crash your cluster with an OutOfMemory (OOM) error if the data size exceeds the driver's memory

Returns a List: It converts distributed data into a standard Python list, where each element is a Row object.

In [0]:
# Create a small DataFrame
df = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])

# Collect all data to the driver
data = df.collect()

# Access data using standard Python indexing
print(data)
print(data[0])
print(data[0][0])
print(data[0][1])
print('___________')
print(data[0]["name"])  # Output: Alice

**collect_list** and **collect_set**

- ```collect_list()``` builds an array from column values, commonly used by groupBy
can be memory intensive

- ```collect_set()```  works the same as collect_list but builds an array of unique values only.
Use this when dupes are not needed and the order does not matter.  This is a less memory intensive action because dupes are removed and not held in memory, also less shuffle overhead for the same reason

![](/Volumes/workspace/pyspark_learning/raw_files/images/collections.png)

![](/Volumes/workspace/pyspark_learning/raw_files/images/best_practices_for_complex_data.png)

## Broadcast on joins

In [0]:
# The broadcast hint forces Spark to broadcast the specified DataFrame during a join, optimizing performance for joins with small tables.
# The network shuffle phase is eliminated

from pyspark.sql.functions import broadcast

# Example: broadcasting df2 in a join with df1
# NOTE: in a broadcast, the small df comes second, not first
broadcast_result = large_df.join(broadcast(small_df), on="id", how="inner")
display(broadcast_result)

### Take

- Take is a method which retrieves the first `n` rows from dataframe.
- Ideal for previewing data
- Simliar to `head`

example:
get the first 5 records from df_2 where storedId equals 25
```
df_2.filter(col("storedId") == 25).take(5)
```

###Explicit Schemas

The following code examples have the same net results


```
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define the custom schema
schema1 = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])
```


Example with spark.read.schema
```
df = spark.read.schema(schema1).csv("path/to/file.csv")
```


Example with spark.read.csv
```
df = spark.read.csv("path/to/file.csv", schema=schema1)
```


### Filter vs Projection

Filtering affects row-wise operations while projection affects column-wise operations, both impacting different aspects of data distribution and network transfer

## Window Functions ##